# Campanhas persistentes de modelos

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="05_treinamento_campanha"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.abdb8f09',
 'model_version': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': True,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'campaign-abdb8f09',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'create',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': True,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Catálogo de campanhas e orçamento. Identidades hi.demo são experimentos sintéticos separados sobre o snapshot, não 500 fontes acadêmicas reais.

In [2]:
display(pd.DataFrame(call(cfg,"GET","/v1/training/campaigns")))
from hashlib import sha256
prefix=sha256(params.request_key.encode()).hexdigest()[:10]
model_keys=[f"hi.demo.campaign.{prefix}.{i:03}" for i in range(params.model_count)]
display({"model_count":len(model_keys),"max_parallel_jobs":params.parallel_jobs,"first_keys":model_keys[:5]})


,campaign_id,status,requested_count,created_at_utc,correlation_id
0,00b98583-e68e-4174-aebc-d7cf2dd03960,COMPLETED,1,2026-11-04T16:29:09.226678+00:00,053f60f5-3198-4212-9720-b92ee91ec3b4
1,52eaaa10-4fa7-4410-861f-71ef572fcb89,RUNNING,500,2026-11-04T16:23:52.504134+00:00,f72c3e08-0717-4138-8b8d-a64b40e8bb36
2,3950bdc3-fb0c-4590-b169-60362016370e,COMPLETED,30,2026-11-04T16:19:33.923367+00:00,16013c03-90fc-45bf-9731-c73ec8a93d6e
3,60a694f5-d771-4d3b-ad99-096ac6c7b101,COMPLETED,1,2026-11-04T16:15:37.671198+00:00,da910fa5-0f94-43de-b904-18c8cbfe5613


{'model_count': 30,
 'max_parallel_jobs': 2,
 'first_keys': ['hi.demo.campaign.83fa031898.000',
  'hi.demo.campaign.83fa031898.001',
  'hi.demo.campaign.83fa031898.002',
  'hi.demo.campaign.83fa031898.003',
  'hi.demo.campaign.83fa031898.004']}

2. Controle opt-in: pause/resume/cancel/retry-failed. Cancelamento é cooperativo; itens em execução concluem, pendentes são cancelados.

In [3]:
if params.campaign_action not in ("create","observe"):
    params.authorize()
    if not params.campaign_id: raise ValueError("campaign_id obrigatório")
    evidence["control"]=call(cfg,"POST",f"/v1/training/campaigns/{params.campaign_id}/{params.campaign_action}")
    display(evidence["control"])


3. Criar campanha somente quando campaign_action=create, ou observar o ID escolhido. Estado/checkpoints permanecem no PostgreSQL após fechar o notebook.

In [4]:
if params.campaign_action!="create":
    params=params.model_copy(update={"execute_write":False})
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


{'preflight': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'cycle_code': '2026-1',
  'model_keys': ['hi.demo.campaign.83fa031898.000',
   'hi.demo.campaign.83fa031898.001',
   'hi.demo.campaign.83fa031898.002',
   'hi.demo.campaign.83fa031898.003',
   'hi.demo.campaign.83fa031898.004',
   'hi.demo.campaign.83fa031898.005',
   'hi.demo.campaign.83fa031898.006',
   'hi.demo.campaign.83fa031898.007',
   'hi.demo.campaign.83fa031898.008',
   'hi.demo.campaign.83fa031898.009',
   'hi.demo.campaign.83fa031898.010',
   'hi.demo.campaign.83fa031898.011',
   'hi.demo.campaign.83fa031898.012',
   'hi.demo.campaign.83fa031898.013',
   'hi.demo.campaign.83fa031898.014',
   'hi.demo.campaign.83fa031898.015',
   'hi.demo.campaign.83fa031898.016',
   'hi.demo.campaign.83fa031898.017',
   'hi.demo.campaign.83fa031898.018',
   'hi.demo.campaign.83fa031898.019',
   'hi.demo.campaign.83fa031898.020',
   'hi.demo.campaign.83fa031898.021',
   'hi.demo.campaign.83fa031898.022',
   'hi.demo.camp

{'status': 'RUNNING', 'counts': {'PENDING': 30}, 'utc': '2026-11-04T16:29:30.801365+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 2, 'PENDING': 28}, 'utc': '2026-11-04T16:29:32.816900+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 4, 'PENDING': 25, 'RUNNING': 1}, 'utc': '2026-11-04T16:29:34.832630+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 6, 'PENDING': 23, 'RUNNING': 1}, 'utc': '2026-11-04T16:29:36.850141+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 8, 'PENDING': 21, 'RUNNING': 1}, 'utc': '2026-11-04T16:29:38.868471+00:00'}


... (10 outputs left out)

{'status': 'RUNNING', 'counts': {'COMPLETED': 28, 'PENDING': 2}, 'utc': '2026-11-04T16:30:01.090324+00:00'}


{'status': 'COMPLETED', 'counts': {'COMPLETED': 30}, 'utc': '2026-11-04T16:30:03.108233+00:00'}


{'campaign_id': '4856c5b5-fe23-4569-87dc-39186a655ed1',
 'idempotency_key': 'campaign-abdb8f09',
 'request_hash': '1ba0d80d54a287162a61f2f58405419f8e38bfb5d84222fa36bbf58ea8d83981',
 'request': {'alpha': 1.0,
  'algorithm': 'ridge',
  'cycle_code': '2026-1',
  'model_keys': ['hi.demo.campaign.83fa031898.000',
   'hi.demo.campaign.83fa031898.001',
   'hi.demo.campaign.83fa031898.002',
   'hi.demo.campaign.83fa031898.003',
   'hi.demo.campaign.83fa031898.004',
   'hi.demo.campaign.83fa031898.005',
   'hi.demo.campaign.83fa031898.006',
   'hi.demo.campaign.83fa031898.007',
   'hi.demo.campaign.83fa031898.008',
   'hi.demo.campaign.83fa031898.009',
   'hi.demo.campaign.83fa031898.010',
   'hi.demo.campaign.83fa031898.011',
   'hi.demo.campaign.83fa031898.012',
   'hi.demo.campaign.83fa031898.013',
   'hi.demo.campaign.83fa031898.014',
   'hi.demo.campaign.83fa031898.015',
   'hi.demo.campaign.83fa031898.016',
   'hi.demo.campaign.83fa031898.017',
   'hi.demo.campaign.83fa031898.018',
   'h

,job_id,campaign_id,model_key,status,attempt,heartbeat_at_utc,started_at_utc,ended_at_utc,mlflow_run_id,registered_model_name,model_version,error_code,result
0,9c92e813-62c9-49ce-8910-ff65a427bcb8,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.000,COMPLETED,1,2026-11-04T16:29:45.350135+00:00,2026-11-04T16:29:44.626306+00:00,2026-11-04T16:29:45.530199+00:00,4bd8a5c80db14bfaa760796fbf2894d0,hi.demo.campaign.83fa031898.000,1,None,"{'run_id': '4bd8a5c80db14bfaa760796fbf2894d0',..."
1,b9c623e4-ea68-4f4b-a020-f6912b159feb,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.001,COMPLETED,1,2026-11-04T16:29:49.333506+00:00,2026-11-04T16:29:48.432634+00:00,2026-11-04T16:29:49.503710+00:00,086155f186324c0c95ad100979a1d1cc,hi.demo.campaign.83fa031898.001,1,None,"{'run_id': '086155f186324c0c95ad100979a1d1cc',..."
2,666fcab0-53c8-4145-9615-52a0f0b36dc4,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.002,COMPLETED,1,2026-11-04T16:29:39.266366+00:00,2026-11-04T16:29:38.373207+00:00,2026-11-04T16:29:39.442847+00:00,27d3f2f2162e4bdb885f2fe1cc48847f,hi.demo.campaign.83fa031898.002,1,None,"{'run_id': '27d3f2f2162e4bdb885f2fe1cc48847f',..."
3,5ef777e7-8c31-4a43-9a55-aeca6c8961a3,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.003,COMPLETED,1,2026-11-04T16:29:37.611356+00:00,2026-11-04T16:29:36.868740+00:00,2026-11-04T16:29:37.801851+00:00,f8be6b9d9ee74b46a9546ad0505ce65b,hi.demo.campaign.83fa031898.003,1,None,"{'run_id': 'f8be6b9d9ee74b46a9546ad0505ce65b',..."
4,0dd42a55-163d-43bf-bad7-a1acab1a40ef,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.004,COMPLETED,1,2026-11-04T16:29:31.744546+00:00,2026-11-04T16:29:30.926135+00:00,2026-11-04T16:29:31.915157+00:00,e1b6eb5439ed4b679636d6d7ce514cab,hi.demo.campaign.83fa031898.004,1,None,"{'run_id': 'e1b6eb5439ed4b679636d6d7ce514cab',..."
5,d8306e26-c448-4dc4-930a-f78ee692d658,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.005,COMPLETED,1,2026-11-04T16:29:55.649489+00:00,2026-11-04T16:29:54.765537+00:00,2026-11-04T16:29:55.932819+00:00,5dfdec868dfb4aefa5f95552a7cfa8d8,hi.demo.campaign.83fa031898.005,1,None,"{'run_id': '5dfdec868dfb4aefa5f95552a7cfa8d8',..."
6,8796b6d2-5ac2-4a42-b0a0-08e65f02cc77,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.006,COMPLETED,1,2026-11-04T16:29:43.880323+00:00,2026-11-04T16:29:43.130971+00:00,2026-11-04T16:29:44.072347+00:00,c22790ce77954a71811524d8891b5c04,hi.demo.campaign.83fa031898.006,1,None,"{'run_id': 'c22790ce77954a71811524d8891b5c04',..."
7,11f17f30-1c93-4a25-8123-8b0f6883ae6e,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.007,COMPLETED,1,2026-11-04T16:29:32.206866+00:00,2026-11-04T16:29:31.424573+00:00,2026-11-04T16:29:32.382243+00:00,4fa610b55e16474ab1770983a50cd660,hi.demo.campaign.83fa031898.007,1,None,"{'run_id': '4fa610b55e16474ab1770983a50cd660',..."
8,83b671c5-240f-4669-b590-e444dc1ae798,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.008,COMPLETED,1,2026-11-04T16:29:41.976585+00:00,2026-11-04T16:29:41.221137+00:00,2026-11-04T16:29:42.179126+00:00,1b0d3a931bd84447add5cadb0614566d,hi.demo.campaign.83fa031898.008,1,None,"{'run_id': '1b0d3a931bd84447add5cadb0614566d',..."
9,872e2351-73cb-45c2-b6e1-0d8e6806bb37,4856c5b5-fe23-4569-87dc-39186a655ed1,hi.demo.campaign.83fa031898.009,COMPLETED,1,2026-11-04T16:29:43.396461+00:00,2026-11-04T16:29:42.703334+00:00,2026-11-04T16:29:43.583167+00:00,1bf968b6ce5049f2b3009514d5f3e14e,hi.demo.campaign.83fa031898.009,1,None,"{'run_id': '1bf968b6ce5049f2b3009514d5f3e14e',..."


4. Quantidades comprovadas e progresso. Treinos reais, versões e instâncias de serving são números distintos.

In [5]:
items=evidence.get("items",[])
display({"requested":len(items),"completed":sum(i["status"]=="COMPLETED" for i in items),"registered_versions":sum(bool(i["model_version"]) for i in items),"automatically_promoted":0,"new_HTTP_servers":0})


{'requested': 30,
 'completed': 30,
 'registered_versions': 30,
 'automatically_promoted': 0,
 'new_HTTP_servers': 0}

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [6]:
display(report(NOTEBOOK_ID,evidence,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação"]))


'/reports/v2/05_treinamento_campanha.json'